[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/17_sequence_and_state_space.ipynb)

# 17. Sequence recurrence and state-space updates — from SSM to Mamba

Mamba의 핵심 계산을 **continuous SSM → discretization → associative scan → input-dependent Δ/B/C → causal depthwise convolution + selective scan + gate** 순서로 본다.

이 버전에서는 causal convolution을 명시적인 left padding으로 구현해서 미래 token이 convolution에 섞이지 않게 한다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Linear state-space recurrence

discrete SSM은 `h_t=A_bar h_{t-1}+B_bar x_t`, `y_t=C h_t+D x_t`로 fixed-size state를 갱신한다. attention처럼 과거 모든 token을 직접 보관할 필요가 없다.


In [ ]:
inputs = torch.tensor([1.0, 0.0, -1.0, 0.5], device=device)
A_bar = torch.tensor(
    [[0.8, 0.1], [0.0, 0.9]],
    device=device,
)
B_bar = torch.tensor([1.0, 0.5], device=device)
C = torch.tensor([0.7, -0.2], device=device)
D = torch.tensor(0.1, device=device)

state = torch.zeros(2, device=device)
outputs = []

for x_t in inputs:
    state = A_bar @ state + B_bar * x_t
    y_t = C @ state + D * x_t
    outputs.append(y_t)

print("outputs:", torch.stack(outputs))


## 2. Continuous SSM → discrete transition

continuous equation `dh/dt=A h+B x`를 token별 step size `Δ`로 discretize하면 diagonal `A`의 경우 핵심 transition은 `exp(ΔA)`로 나타난다.


In [ ]:
A = torch.tensor([-1.0, -2.0], device=device)
B = torch.tensor([1.0, 0.5], device=device)
delta = torch.tensor(0.25, device=device)

A_discrete = torch.exp(delta * A)
B_input = delta * B

print("exp(delta*A):", A_discrete)
print("delta*B:", B_input)


## 3. Associative scan viewpoint

affine recurrence `h' = a h + b`는 pair `(a,b)`의 associative composition으로 묶을 수 있어 parallel scan이 가능하다.


In [ ]:
a1 = torch.tensor(0.5, device=device)
b1 = torch.tensor(1.0, device=device)
a2 = torch.tensor(0.8, device=device)
b2 = torch.tensor(2.0, device=device)

a_composed = a2 * a1
b_composed = a2 * b1 + b2

h0 = torch.tensor(3.0, device=device)
sequential = a2 * (a1 * h0 + b1) + b2
composed = a_composed * h0 + b_composed

print("sequential:", sequential)
print("composed:", composed)


## 4. Selective scan: Δ, B, C depend on the token

Mamba의 선택성은 단순 sigmoid gate가 아니라 각 token에서 `Δ_t`, `B_t`, `C_t`를 만들어 state transition, write, read를 입력 의존적으로 바꾸는 데 있다.


In [ ]:
def selective_scan_reference(u, delta, A, B, C, D=None):
    batch_size, channels, sequence_length = u.shape
    state_dim = A.size(1)

    state = torch.zeros(
        batch_size,
        channels,
        state_dim,
        device=u.device,
        dtype=u.dtype,
    )
    outputs = []

    delta_A = torch.exp(
        delta[:, :, :, None]
        * A[None, :, None, :]
    )

    for time_index in range(sequence_length):
        delta_t = delta[:, :, time_index]
        u_t = u[:, :, time_index]
        B_t = B[:, time_index, :]
        C_t = C[:, time_index, :]

        input_to_state = (
            delta_t[:, :, None]
            * B_t[:, None, :]
            * u_t[:, :, None]
        )

        state = (
            delta_A[:, :, time_index, :] * state
            + input_to_state
        )

        y_t = torch.sum(
            state * C_t[:, None, :],
            dim=-1,
        )

        if D is not None:
            y_t = y_t + D[None, :] * u_t

        outputs.append(y_t)

    return torch.stack(outputs, dim=-1)


batch_size = 1
channels = 3
sequence_length = 5
state_dim = 2

u = torch.randn(
    batch_size, channels, sequence_length,
    device=device,
)
features = u.transpose(1, 2)

delta_projection = nn.Linear(channels, channels).to(device)
B_projection = nn.Linear(channels, state_dim).to(device)
C_projection = nn.Linear(channels, state_dim).to(device)

delta = F.softplus(delta_projection(features)).transpose(1, 2)
B_variable = B_projection(features)
C_variable = C_projection(features)

A_log = torch.log(
    torch.arange(1, state_dim + 1, device=device)
    .float()
    .repeat(channels, 1)
)
A = -torch.exp(A_log)
D = torch.ones(channels, device=device)

selective_output = selective_scan_reference(
    u, delta, A, B_variable, C_variable, D
)

print("delta:", delta.shape)
print("B_t:", B_variable.shape)
print("C_t:", C_variable.shape)
print("output:", selective_output.shape)


## 5. Tiny Mamba-style block with truly causal depthwise convolution

input projection은 `u` branch와 `z` gate branch를 만든다. `u` branch는 **왼쪽만 padding한 causal depthwise convolution**을 통과한 뒤 selective scan으로 들어가고, scan output은 `SiLU(z)`와 곱해져 output projection으로 간다.


In [ ]:
class TinyMambaBlock(nn.Module):
    def __init__(
        self,
        model_dim=8,
        inner_dim=12,
        state_dim=4,
        conv_kernel=3,
    ):
        super().__init__()

        self.inner_dim = inner_dim
        self.state_dim = state_dim
        self.conv_kernel = conv_kernel

        self.input_projection = nn.Linear(
            model_dim,
            2 * inner_dim,
        )
        self.depthwise_conv = nn.Conv1d(
            inner_dim,
            inner_dim,
            kernel_size=conv_kernel,
            padding=0,
            groups=inner_dim,
        )

        self.delta_projection = nn.Linear(inner_dim, inner_dim)
        self.B_projection = nn.Linear(inner_dim, state_dim)
        self.C_projection = nn.Linear(inner_dim, state_dim)

        initial_A = torch.arange(1, state_dim + 1).float()
        initial_A = initial_A.repeat(inner_dim, 1)
        self.A_log = nn.Parameter(torch.log(initial_A))
        self.D = nn.Parameter(torch.ones(inner_dim))

        self.output_projection = nn.Linear(inner_dim, model_dim)

    def forward(self, x):
        projected = self.input_projection(x)
        u_branch, z_branch = projected.chunk(2, dim=-1)

        u = u_branch.transpose(1, 2)
        left_padding = self.conv_kernel - 1
        u_padded = F.pad(u, (left_padding, 0))
        convolved = self.depthwise_conv(u_padded)
        convolved = F.silu(convolved)

        features = convolved.transpose(1, 2)
        delta = F.softplus(
            self.delta_projection(features)
        ).transpose(1, 2)
        B = self.B_projection(features)
        C = self.C_projection(features)
        A = -torch.exp(self.A_log)

        scanned = selective_scan_reference(
            convolved,
            delta,
            A,
            B,
            C,
            self.D,
        )

        gated = (
            scanned.transpose(1, 2)
            * F.silu(z_branch)
        )
        return self.output_projection(gated)


block = TinyMambaBlock().to(device)
sequence = torch.randn(2, 6, 8, device=device)
output = block(sequence)

print("input:", sequence.shape)
print("output:", output.shape)


## 6. Causality sanity check

미래 token만 바꿨을 때 이전 위치의 convolution output이 바뀌지 않는지 직접 확인한다.


In [ ]:
test_conv = nn.Conv1d(
    1, 1,
    kernel_size=3,
    padding=0,
    bias=False,
).to(device)

sequence_a = torch.tensor([[[1.0, 2.0, 3.0, 4.0]]], device=device)
sequence_b = sequence_a.clone()
sequence_b[:, :, -1] = 100.0

def causal_conv(x):
    return test_conv(F.pad(x, (2, 0)))

output_a = causal_conv(sequence_a)
output_b = causal_conv(sequence_b)

print(
    "past outputs unchanged:",
    torch.allclose(output_a[:, :, :-1], output_b[:, :, :-1]),
)


## References and provenance

**S4** — Gu et al. continuous/discrete state-space viewpoint와 scan 구조를 참조했다.

**Mamba** — Gu & Dao 및 공식 `state-spaces/mamba` selective scan reference. `deltaA=exp(delta*A)`, input-dependent `delta/B/C`, `D*u`, input split, causal depthwise convolution, SiLU gating, output projection을 작은 형태로 보존했다.
